<a href="https://colab.research.google.com/github/yashb98/90Days_Machine_learinng/blob/main/Intro_to_LLMs_Building_a_RAG_system_(Project_9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Retrieval-Augmented Generation (RAG) is a method that combines retrieval and generation:

**(1) Retrieval**

•	This is fetching relevant information from your own documents.

•	Done using vector embeddings + vector database.

•	Helps the model focus on relevant information instead of generating purely from training data.

**(2) Generation**


•	Uses a large language model (LLM) like GPT to generate answers based on retrieved content.

•	Ensures the answers are grounded in actual data, reducing hallucinations.

**Why it matters:**

•	A regular LLM can make up information.

•	RAG ensures answers are backed by your own documents, making it suitable for research papers, manuals, or any custom knowledge base.

## Install Dependencies


### This cell ensures all necessary tools are available for building the pipeline by installing or updating required Python packages:

1. **langchain:** A framework for simplifying the creation of applications that use LLMs by chaining together various components.

2. **langchain-community:** Provides integrations for external third-party resources, such as document loaders, vector stores, and specific model bindings.

3. **langchain-text-splitters:** A utility library dedicated to breaking down large documents into smaller, manageable chunks. This addresses the LLM's context length limitation.

4. **sentence-transformers:** A specialized library that provides models to convert text into fixed-size numerical vectors, known as embeddings. These embeddings are vital for the semantic search component of the Retrieval step.

5. **faiss-cpu:** A library by Facebook AI for performing efficient similarity search across large datasets of dense vectors, serving as the Vector Database in a RAG pipeline.

6. **pypdf:** A pure-Python tool used to read and extract textual data from PDF files.

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf

## Import Modules

### This imports the necessary classes and functions from the installed libraries:

PyPDFLoader: Instantiated to load the contents of the PDF document.

RecursiveCharacterTextSplitter: The chosen implementation for breaking the input document into chunks.

SentenceTransformerEmbeddings: Used to select and load a sentence transformer model for generating semantic vectors (embeddings).

FAISS: Imported to create the in-memory index for fast similarity search across the vector embeddings.

os: A standard Python module for interfacing with the operating system, often used for file path operations.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
import os

## Load your PDF document

Here I am using a book for builduing AI Engineering applications ( by Chip Huyen )



#### This is the first data processing step, taking the raw file and turning it into data objects:

1. pdf_path = ...: Defines the location of the source document ("AI Engineering" by Chip Huyen).

2. loader = PyPDFLoader(pdf_path): Creates a loader object tied to the document path.

3. pages = loader.load(): Executes the PDF extraction, resulting in a list where each element is a LangChain Document object corresponding to a single page from the PDF.

**Purpose:** The output confirms 991 pages were loaded, verifying successful data access and text extraction.

In [ ]:
# Load PDF pages
pdf_path = "/content/_OceanofPDF.com_AI_Engineering_Building_Applications_-_Chip_Huyen.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Total pages loaded: {len(pages)}")
print("Sample page content:\n")
print(pages[3].page_content[:500])

#### A data inspection step to understand the overall size of the source text:

* full_text = " ".join(...): Iterates through the loaded pages and concatenates all the text into a single string.

**Purpose:** The output shows the raw document size is 1,080,309 characters. This confirms the content is too large for most LLM context windows, mandating the need for the chunking step that follows.

In [ ]:
# Combine all text for exploration
full_text = " ".join([p.page_content for p in pages])
print(f"Document length: {len(full_text)} characters")
print(f"Sample snippet:\n{full_text[:600]}")

## Split the document into Chunks

### This is the essential preparation for the retrieval step, optimizing the data for vector search:

1. splitter = RecursiveCharacterTextSplitter(...): Initializes the text splitting strategy.

2. chunk_size=1000: Sets the maximum size for each unit of text (chunk) at 1,000 characters.

3. chunk_overlap=200: Ensures the last 200 characters of one chunk are included in the start of the next chunk. This is a best practice to preserve context continuity and improve retrieval accuracy.

4. chunks = splitter.split_documents(pages): Executes the splitting, creating the final list of smaller, context-aware document chunks.

**Purpose:** The result, 1,640 chunks, transforms the single large document into a large set of semantically meaningful, searchable units.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(pages)

print(f" Created {len(chunks)} chunks from {len(pages)} pages.")
print(chunks[0].page_content[:400])

In [ ]:
print(f"Total chunks: {len(chunks)}")

### Access a specific chunk

#### These final cells serve only to confirm the data preparation steps succeeded:

* Accessing/Printing Chunks: Prints the first 400 characters of the first chunk, the total number of chunks, and the content of specific chunks by index (e.g., index 2, index 4, index 12).

**Purpose:** To verify that the chunks array is correctly populated and contains clean, readable, segmented text, ensuring the data is ready for the subsequent RAG steps (embedding and indexing).


In [ ]:
# First chunk
print(chunks[0].page_content)

print("\n")

# Fifth chunk
print(chunks[4].page_content)

print("\n")

# 13th chunk
print(chunks[12].page_content)

### Loop Through all chunks

If you want to quickly see the first 200 characters of each chunk:

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content[:200])
    print("\n")
    if i == 9:
        break

### Access a chunk for later use

In [ ]:
example_chunk = chunks[2].page_content
print(example_chunk)

## Create Embeddings

We’ll use a pre-trained sentence embedding model from Hugging Face — "sentence-transformers/all-MiniLM-L6-v2" (lightweight and fast).

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Test embedding creation on one sample chunk
test_embedding = embedding_model.embed_query(chunks[0].page_content)
print(f" Sample embedding vector length: {len(test_embedding)}")

#### We’ll encode each chunk into a vector representation.

In [ ]:
texts = [c.page_content for c in chunks]
embeddings = embedding_model.embed_documents(texts)

print(" Generated embeddings with shape:", len(embeddings), "x", len(embeddings[0]))

#### embed_documents() returns a list of embeddings rather than a NumPy array, so before adding to FAISS you’ll need:



In [ ]:
import numpy as np
embeddings = np.array(embeddings, dtype = "float32")

## Build a FAISS Vector Store

FAISS (Facebook AI Similarity Search) helps us store and quickly find similar vectors (chunks).

In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # inner product = cosine similarity for normalized vectors
index.add(embeddings.astype("float32"))

print(f" FAISS index created with {index.ntotal} vectors.")

## Save the FAISS Index & Metadata

We’ll store:

	•	The FAISS binary index file
	•	The chunk metadata (page numbers, chunk IDs, etc.)

In [ ]:
import json

os.makedirs("faiss_index_ai_engineering", exist_ok=True)
faiss.write_index(index, "faiss_index_ai_engineering/index.faiss")

metadata = [c.metadata for c in chunks]
with open("faiss_index_ai_engineering/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(" Saved FAISS index and metadata locally.")

## Test Semantic Search

Let’s test if your RAG base is working properly.

In [ ]:
def semantic_search(query, k=3):
    # Use LangChain method for embeddings
    query_emb = embedding_model.embed_query(query)
    query_emb = np.array([query_emb], dtype="float32")

    # Perform FAISS search
    distances, indices = index.search(query_emb, k)

    # Collect results
    results = []
    for i, idx in enumerate(indices[0]):
        text = chunks[idx].page_content[:400].replace("\n", " ")
        page = chunks[idx].metadata.get("page", "Unknown")
        results.append({"rank": i+1, "page": page, "text": text})

    return results

### Define your queries

These are natural language questions you want to ask your document (in this case, Chip Huyen’s AI Engineering book).

In [ ]:
# Example user queries
queries = [
    "What is AI engineering?",
    "How does the book describe model deployment?",
    "What are some challenges in real-world machine learning systems?",
]

###Run the search for each query

We’ll loop through each question and show the top k (default 3) most relevant chunks.

In [ ]:
for q in queries:
    print(f"\n Query: {q}\n" + "-"*80)
    results = semantic_search(q, k=3)

    for r in results:
        print(f" Rank {r['rank']} | Page {r['page']}")
        print(r['text'])
        print("-"*80)

In [ ]:
!pip install google-generativeai sentence-transformers

In [ ]:
!pip show google-generativeai

## Configure Gemini API key

In [ ]:

from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

## Load FAISS index and chunk data (from Day 23)

In [ ]:
FAISS_INDEX_DIR = "faiss_index_ai_engineering"

# Load FAISS index
index = faiss.read_index(os.path.join(FAISS_INDEX_DIR, "index.faiss"))

# Load metadata
with open(os.path.join(FAISS_INDEX_DIR, "metadata.json"), "r", encoding="utf-8") as f:
    metadatas = json.load(f)

texts = [chunk.page_content for chunk in chunks]

with open(os.path.join(FAISS_INDEX_DIR, "texts.txt"), "w", encoding="utf-8") as f:
    f.write("\n\n".join(texts))
# Load texts
with open(os.path.join(FAISS_INDEX_DIR, "texts.txt"), "r", encoding="utf-8") as f:
    all_texts = f.read().split("\n\n")

print(f"Loaded FAISS index and {len(all_texts)} text chunks.")

 ## Retrieve top relevant chunks for a user query

In [ ]:
def retrieve_chunks(query, k=3):

    query_emb = embedding_model.embed_query(query)
    query_emb = np.array([query_emb], dtype="float32")

    # Search top-k similar chunks in FAISS index
    distances, indices = index.search(query_emb, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(all_texts):
            text = all_texts[idx].replace("\n", " ")
            meta = metadatas[idx]
            results.append({
                "score": float(dist),
                "text": text,
                "page": meta.get("page", "Unknown"),
                "chunk_id": meta.get("chunk_id", idx)
            })
    return results

	•	Converts the query into the same embedding format.
	•	Searches in FAISS for the k most similar chunks.
	•	Returns those chunks + metadata for reference.


## Build a RAG prompt

In [ ]:
import re


def sanitize_text(text: str) -> str:
    """Clean up text to remove problematic characters before sending to Gemini."""
    if not isinstance(text, str):
        text = str(text)
    # Remove leading/trailing whitespace and control characters
    text = text.strip()
    # Replace smart quotes and Unicode dashes
    text = (text
            .replace("“", '"')
            .replace("”", '"')
            .replace("’", "'")
            .replace("–", "-")
            .replace("—", "-"))
    # Remove carriage returns, tabs, and multiple newlines
    text = re.sub(r"[\r\n\t]+", " ", text)
    return text

In [ ]:
def build_prompt(query, retrieved_chunks):
    """Build a clean, safe prompt for Gemini."""
    context_parts = []
    for r in retrieved_chunks:
        safe_text = sanitize_text(r.get("text", ""))
        context_parts.append(f"[Page {r.get('page', '?')}, Chunk {r.get('chunk_id', '?')}]\n{safe_text}")

    context = "\n\n".join(context_parts)
    query = sanitize_text(query)

    prompt = (
        "You are a helpful assistant. Use the provided context to answer the user's question. "
        "If you don't find the answer, look again.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    )
    return prompt

## Generate the answer using Gemini

In [ ]:
def generate_answer(prompt):
    model = genai.GenerativeModel("gemini-2.5-flash")
    # Explicitly create content as a list of parts, ensuring it's treated as text
    content = [
        {
            "text": prompt
        }
    ]
    print(f"Type of content being sent: {type(content)}")
    print(f"Content being sent: {content}")
    response = model.generate_content(content)
    return response.text

## Create a complete RAG pipeline

In [ ]:
def rag_query(query, k=3):
    retrieved = retrieve_chunks(query, k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)

    print(f" Question: {query}\n")
    print(" Answer:", answer, "\n")
    print(" Sources:")
    for r in retrieved:
        print(f"- Page: {r['page']} | Chunk: {r['chunk_id']} | Score: {r['score']:.4f}")

## Test Queries for RAG

In [33]:
# List of 10 test queries
test_queries = [
    "What is AI engineering and how does it differ from traditional software engineering?",
    "How can machine learning models be deployed in production safely?",
    "What are the key components of a scalable ML data pipeline?",
    "How do you monitor ML models in production for performance drift?",
    "What are the best practices for feature engineering in AI projects?",
    "How should AI engineers structure experiments to validate models?",
    "What are the main ethical considerations in AI system design?",
    "Which tools and platforms are recommended for scalable AI infrastructure?",
    "What are common MLOps practices for maintaining ML workflows?",
    "Give examples of real-world applications of AI engineering principles."
]

# Loop through the queries
for i, query in enumerate(test_queries, 1):
    # print(f"\n Test Query {i}: {query}\n") # Removed this line
    rag_query(query, k=3)  # Calls your existing RAG pipeline
    print("------------------------------------------------------------")

Type of content being sent: <class 'list'>
Content being sent: [{'text': "You are a helpful assistant. Use the provided context to answer the user's question. If you don't find the answer, look again.\n\nContext:\n[Page 13, Chunk 17]\nThe first chapter of this book also covers the differences between traditional ML engineering and AI engineering. A real-world system often involves both traditional ML models and foundation models, so knowledge about working with both is often necessary. Determining whether something will last, however, is often challenging. I relied on three criteria. First, for a problem, I determined whether it results from the fundamental limitations of how AI works or if it'll go away with\n\n[Page 91, Chunk 146]\nFigure 1-12. Many companies put AI engineering and ML engineering under the same umbrella, as shown in the job headlines on LinkedIn from December 17, 2023. Some companies have separate job descriptions for AI engineering, as shown in Figure 1-13. Regardle

## Analysis of the Notebook and Results

This notebook successfully implements a basic Retrieval-Augmented Generation (RAG) pipeline using a PDF document. Here's a breakdown of the key steps and observations:

**1. Setup and Dependencies:**

*   The notebook starts by installing necessary libraries (`langchain`, `sentence-transformers`, `faiss-cpu`, `pypdf`, etc.). This is a standard and correct approach for setting up the environment.
*   Importing the required modules is also done correctly.

**2. Data Loading and Preprocessing:**

*   The `PyPDFLoader` is used to load the PDF document. The output shows that 991 pages were loaded, indicating successful file reading.
*   Combining the text from all pages into a single string and checking its length confirms that the document is indeed large and requires splitting.
*   The `RecursiveCharacterTextSplitter` is used to break down the document into smaller chunks with a specified `chunk_size` and `chunk_overlap`. This is a crucial step for managing the context length of the language model and ensuring semantic continuity between chunks. The creation of 1640 chunks seems reasonable for a document of this size.
*   Inspecting individual chunks confirms that the splitting process worked as expected.

**3. Embedding Creation:**

*   The notebook uses `HuggingFaceEmbeddings` with the "sentence-transformers/all-MiniLM-L6-v2" model to create embeddings for each chunk. This is a good choice for its balance of performance and efficiency.
*   The warning about `LangChainDeprecationWarning` for `HuggingFaceEmbeddings` is noted. While the current code still works, it's good practice to update to the recommended `langchain_huggingface` package in the future.
*   Creating and checking the shape of the embeddings confirms that vectors of the expected dimension (384) were generated for all 1640 chunks.
*   Converting the embeddings to `float32` NumPy array is necessary for compatibility with FAISS.

**4. FAISS Vector Store:**

*   A FAISS `IndexFlatIP` is created, which is suitable for cosine similarity search when vectors are normalized (as is the case with Sentence Transformers).
*   Adding the embeddings to the index is done correctly.
*   The confirmation that the FAISS index contains 1640 vectors verifies that all chunks were indexed.

**5. Saving and Loading the Index and Metadata:**

*   Saving the FAISS index and metadata (`metadata.json` and `texts.txt`) is a good practice for persistence, allowing the index to be reused without re-processing the entire document.
*   Loading the saved index, metadata, and texts confirms that the persistence mechanism works.

**6. Semantic Search and RAG Pipeline:**

*   The `semantic_search` function correctly uses the embedding model and the FAISS index to retrieve the top-k most similar chunks based on a query.
*   The `build_prompt` function constructs a prompt for the language model, including the original query and the retrieved context. This is the core of the RAG approach.
*   The `generate_answer` function uses the Gemini 2.5 Flash model to generate a response based on the provided prompt.
*   The `rag_query` function orchestrates the retrieval and generation steps.
*   The initial test query "What are the key principles of AI engineering?" returned "I don’t know from the provided document." while showing relevant chunks in the sources. This suggests that the retrieved chunks might not contain the explicit answer in a way the model can easily extract, or the model's ability to synthesize the answer from the given chunks is limited.
*   The subsequent test queries show mixed results. Some queries (e.g., "How can machine learning models be deployed in production safely?", "How do you monitor ML models in production for performance drift?", "How should AI engineers structure experiments to validate models?", "What are the main ethical considerations in AI system design?") receive relevant answers extracted from the document, indicating that for certain topics, the RAG pipeline works well.
*   Other queries (e.g., "What is AI engineering and how does it differ from traditional software engineering?", "What are the key components of a scalable ML data pipeline?", "What are the best practices for feature engineering in AI projects?", "Which tools and platforms are recommended for scalable AI infrastructure?", "What are common MLOps practices for maintaining ML workflows?", "Give examples of real-world applications of AI engineering principles.") still result in "I don’t know from the provided document." This suggests that either the retrieval is not bringing back the most relevant chunks for these specific questions, or the model is unable to synthesize the answer from the retrieved information.

## Conclusion and Summary

The notebook provides a solid foundation for a RAG pipeline. It successfully loads, preprocesses, embeds, and indexes a large PDF document. The basic semantic search and RAG query functions are implemented correctly.

However, the performance of the RAG pipeline in answering some of the test queries is limited, with several questions resulting in "I don’t know from the provided document." This indicates areas for improvement in either the retrieval or the generation stage of the pipeline.

**Summary:**

*   Basic RAG pipeline successfully implemented.
*   PDF loaded, chunked, embedded, and indexed in FAISS.
*   Semantic search retrieves relevant chunks.
*   Gemini 2.5 Flash model used for generation.
*   Performance on some queries is not optimal, highlighting the need for further refinement.

